# Porównanie Sieci MLP: Float32 vs Fixed-Point (PTQ i QAT)
W tym projekcie badamy wpływ kwantyzacji na rozmiar i dokładność prostej sieci neuronowej trenowanej na zbiorze MNIST.

In [1]:
import os
import copy
import time # NOWE: do mierzenia czasu
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tabulate import tabulate

BATCH_SIZE = 64
FP32_EPOCHS = 5
QAT_EPOCHS = 2
DEVICE = torch.device('cpu')

supported_engines = torch.backends.quantized.supported_engines
if 'x86' in supported_engines:
    Q_ENGINE = 'x86'
elif 'fbgemm' in supported_engines:
    Q_ENGINE = 'fbgemm'
elif 'qnnpack' in supported_engines:
    Q_ENGINE = 'qnnpack'
else:
    Q_ENGINE = supported_engines[0]

torch.backends.quantized.engine = Q_ENGINE
print(f"Wybrano silnik: {Q_ENGINE}")

ModuleNotFoundError: No module named 'torchvision'

In [ ]:
class QuantizedMLP(nn.Module):
    def __init__(self):
        super(QuantizedMLP, self).__init__()
        # QuantStub: wejście -> fixed point
        self.quant = torch.ao.quantization.QuantStub()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(64, 10)
        # DeQuantStub: fixed point -> wyjście (float)
        self.dequant = torch.ao.quantization.DeQuantStub()

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = self.quant(x)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        x = self.dequant(x)
        return x

# Ładowanie danych
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Dane załadowane! Ilość paczek treningowych: {len(train_loader)}")

Dane załadowane! Ilość paczek treningowych: 938


In [ ]:
def train_model(model, train_loader, criterion, optimizer, epochs=1):
    model.train()
    for epoch in range(epochs):
        for data, target in train_loader:
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
        print(f"Zakończono epokę {epoch+1}/{epochs}")

def evaluate_model(model, test_loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            output = model(data)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    return 100. * correct / len(test_loader.dataset)

def print_size_of_model(model):
    torch.save(model.state_dict(), "temp.p")
    size = os.path.getsize("temp.p") / 1e3
    os.remove("temp.p")
    return size

def measure_inference_time(model, test_loader, warmup_batches=20, repeats=3):
    model.eval()

    # Rozgrzewka — więcej batchy dla stabilności CPU
    with torch.no_grad():
        for i, (data, _) in enumerate(test_loader):
            _ = model(data)
            if i >= warmup_batches:
                break

    # Kilka pełnych przebiegów — bierzemy minimum (eliminuje zakłócenia OS)
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        with torch.no_grad():
            for data, _ in test_loader:
                _ = model(data)
        times.append(time.perf_counter() - start)

    return min(times) * 1000  # ms

### Eksperyment 1: Sieć bazowa Float32 (FP32)
Trenujemy standardową sieć bez żadnych modyfikacji.

In [20]:
criterion = nn.CrossEntropyLoss()

print("--- Trenowanie modelu bazowego (Float 32) ---")
model_fp32 = QuantizedMLP().to(DEVICE)
optimizer = optim.Adam(model_fp32.parameters(), lr=0.001)

train_model(model_fp32, train_loader, criterion, optimizer, epochs=FP32_EPOCHS)

acc_fp32 = evaluate_model(model_fp32, test_loader)
size_fp32 = print_size_of_model(model_fp32)
print(f"FP32 Dokładność: {acc_fp32:.2f}%, Rozmiar: {size_fp32:.2f} KB")

--- Trenowanie modelu bazowego (Float 32) ---
Zakończono epokę 1/5
Zakończono epokę 2/5
Zakończono epokę 3/5
Zakończono epokę 4/5
Zakończono epokę 5/5
FP32 Dokładność: 97.83%, Rozmiar: 440.24 KB


### Eksperyment 2: Post-Training Quantization (PTQ)
Kopiujemy wytrenowany model FP32, kalibrujemy go na danych testowych i konwertujemy jego wagi na postać stałoprzecinkową (INT8).

In [21]:
print("--- Post-Training Quantization (PTQ) ---")
model_ptq = copy.deepcopy(model_fp32)
model_ptq.eval()

# Konfiguracja i przygotowanie
model_ptq.qconfig = torch.ao.quantization.get_default_qconfig(Q_ENGINE)
torch.ao.quantization.prepare(model_ptq, inplace=True)

# Kalibracja
print("Kalibracja modelu PTQ...")
calibration_dataset = torch.utils.data.Subset(train_dataset, indices=range(2000))
calibration_loader = DataLoader(calibration_dataset, batch_size=BATCH_SIZE, shuffle=False)

evaluate_model(model_ptq, calibration_loader)

# Konwersja na fixed-point
torch.ao.quantization.convert(model_ptq, inplace=True)

acc_ptq = evaluate_model(model_ptq, test_loader)
size_ptq = print_size_of_model(model_ptq)
print(f"PTQ Dokładność: {acc_ptq:.2f}%, Rozmiar: {size_ptq:.2f} KB")

--- Post-Training Quantization (PTQ) ---
Kalibracja modelu PTQ...


/tmp/ipykernel_4210/2117507139.py:7: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  torch.ao.quantization.prepare(model_ptq, inplace=True)
/usr/local/lib/python3.12/dist-packages/torch/ao/quantization/observer.py:1039: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release

PTQ Dokładność: 97.82%, Rozmiar: 119.45 KB


### Eksperyment 3: Quantization-Aware Training (QAT)
Kopiujemy bazowy model, ale poddajemy go dalszemu trenowaniu symulując obcięcie precyzji w trakcie wstecznej propagacji błędu.

In [22]:
print("--- Quantization-Aware Training (QAT) ---")
model_qat = copy.deepcopy(model_fp32)
model_qat.train()

# Konfiguracja
model_qat.qconfig = torch.ao.quantization.get_default_qat_qconfig(Q_ENGINE)
torch.ao.quantization.prepare_qat(model_qat, inplace=True)

# Dotrenowywanie (Fake Quantization) z mniejszym learning rate
print("Dotrenowywanie modelu QAT...")
optimizer_qat = optim.Adam(model_qat.parameters(), lr=0.0001)
train_model(model_qat, train_loader, criterion, optimizer_qat, epochs=QAT_EPOCHS)

# Konwersja na gotowy model
model_qat.eval()
torch.ao.quantization.convert(model_qat, inplace=True)

acc_qat = evaluate_model(model_qat, test_loader)
size_qat = print_size_of_model(model_qat)
print(f"QAT Dokładność: {acc_qat:.2f}%, Rozmiar: {size_qat:.2f} KB")

--- Quantization-Aware Training (QAT) ---
Dotrenowywanie modelu QAT...


/tmp/ipykernel_4210/4193967697.py:7: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  torch.ao.quantization.prepare_qat(model_qat, inplace=True)
/usr/local/lib/python3.12/dist-packages/torch/ao/quantization/observer.py:534: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future rele

Zakończono epokę 1/2
Zakończono epokę 2/2


/tmp/ipykernel_4210/4193967697.py:16: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  torch.ao.quantization.convert(model_qat, inplace=True)


QAT Dokładność: 98.11%, Rozmiar: 119.45 KB


### Podsumowanie i Wnioski
Zestawienie spadku dokładności w stosunku do zysku na rozmiarze modelu w pamięci.

In [23]:
time_fp32 = measure_inference_time(model_fp32, test_loader)
time_ptq = measure_inference_time(model_ptq, test_loader)
time_qat = measure_inference_time(model_qat, test_loader)

results = [
    ["Float32 (Baza)", f"{acc_fp32:.2f}%", f"{size_fp32:.2f} KB", f"{time_fp32:.1f} ms"],
    ["PTQ (Post-Training)", f"{acc_ptq:.2f}%", f"{size_ptq:.2f} KB", f"{time_ptq:.1f} ms"],
    ["QAT (Aware Training)", f"{acc_qat:.2f}%", f"{size_qat:.2f} KB", f"{time_qat:.1f} ms"]
]
print(tabulate(results, headers=["Model Typ", "Dokładność", "Rozmiar Pamięci", "Czas Inferencji (10k obr.)"], tablefmt="grid"))

+----------------------+--------------+-------------------+------------------------------+
| Model Typ            | Dokładność   | Rozmiar Pamięci   | Czas Inferencji (10k obr.)   |
+======================+==============+===================+==============================+
| Float32 (Baza)       | 97.83%       | 440.24 KB         | 1996.3 ms                    |
+----------------------+--------------+-------------------+------------------------------+
| PTQ (Post-Training)  | 97.82%       | 119.45 KB         | 1981.4 ms                    |
+----------------------+--------------+-------------------+------------------------------+
| QAT (Aware Training) | 98.11%       | 119.45 KB         | 2012.8 ms                    |
+----------------------+--------------+-------------------+------------------------------+


In [27]:
def measure_generalization_gap(model, train_loader, test_loader):
    """
    Mierzy lukę generalizacyjną (Train Acc - Test Acc).
    Mniejsza luka oznacza lepszą regularyzację (mniejszy overfitting),
    ponieważ model radzi sobie z nowymi danymi niemal tak samo dobrze jak z treningowymi.
    """
    # Używamy Twojej istniejącej funkcji evaluate_model
    train_acc = evaluate_model(model, train_loader)
    test_acc = evaluate_model(model, test_loader)

    # Obliczanie luki (gap)
    gap = train_acc - test_acc

    return train_acc, test_acc, gap

def evaluate_with_noise(model, test_loader, noise_std=0.5):
    """
    Testuje model na danych, do których dodano szum Gaussa.
    Większa odporność na szum = lepsza regularyzacja.
    """
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            # Generowanie szumu i dodawanie do oryginalnych obrazków
            noise = torch.randn_like(data) * noise_std
            noisy_data = data + noise

            output = model(noisy_data)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    return 100. * correct / len(test_loader.dataset)

def evaluate_test_loss(model, test_loader, criterion):
    """
    Mierzy błąd (Cross-Entropy Loss) na zbiorze testowym.
    Niższy błąd oznacza, że model nie jest "zbyt pewny siebie" przy błędnych predykcjach.
    """
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for data, target in test_loader:
            output = model(data)
            loss = criterion(output, target)
            total_loss += loss.item() * data.size(0)

    return total_loss / len(test_loader.dataset)

def compare_advanced_regularization(model_fp32, model_ptq, model_qat, test_loader,train_loader, criterion):
    print("\n--- Zaawansowana Analiza Regularyzacji ---")

    # Parametr szumu - 0.5 to dość silne zniekształcenie dla MNIST
    NOISE_LEVEL = 0.5

    # 1. Test na szum
    noise_acc_fp32 = evaluate_with_noise(model_fp32, test_loader, noise_std=NOISE_LEVEL)
    noise_acc_ptq = evaluate_with_noise(model_ptq, test_loader, noise_std=NOISE_LEVEL)
    noise_acc_qat = evaluate_with_noise(model_qat, test_loader, noise_std=NOISE_LEVEL)

    # 2. Test Loss
    loss_fp32 = evaluate_test_loss(model_fp32, test_loader, criterion)
    loss_ptq = evaluate_test_loss(model_ptq, test_loader, criterion)
    loss_qat = evaluate_test_loss(model_qat, test_loader, criterion)

    _,_,train_test_fp32 = measure_generalization_gap(model_fp32, train_loader, test_loader)
    _,_,train_test_ptq = measure_generalization_gap(model_ptq, train_loader, test_loader)
    _,_,train_test_qat = measure_generalization_gap(model_qat, train_loader, test_loader)
    results = [
        ["Float32 (Baza)", f"{noise_acc_fp32:.2f}%", f"{loss_fp32:.4f}", f"{train_test_fp32:.4f}"],
        ["PTQ (Post-Training)", f"{noise_acc_ptq:.2f}%", f"{loss_ptq:.4f}", f"{train_test_ptq:.4f}"],
        ["QAT (Aware Training)", f"{noise_acc_qat:.2f}%", f"{loss_qat:.4f}", f"{train_test_qat:.4f}"]
    ]

    print(f"\nDodano szum Gaussa (std={NOISE_LEVEL}) do zbioru testowego.")
    print("Wyjaśnienie:")
    print("- 'Acc na szumie': Im wyższa wartość, tym bardziej odporny i zregularyzowany model.")
    print("- 'Test Loss': Im MNIEJSZA wartość, tym model rzadziej popełnia 'pewne siebie' błędy.")
    print(tabulate(results, headers=["Model Typ", "Acc na szumie", "Test Loss", "GAP Train vs Test"], tablefmt="grid"))

# Wywołanie nowej funkcji
# QAT poprawil zdolnosc do regularyzacji
compare_advanced_regularization(model_fp32, model_ptq, model_qat, test_loader, train_loader, criterion)


--- Zaawansowana Analiza Regularyzacji ---

Dodano szum Gaussa (std=0.5) do zbioru testowego.
Wyjaśnienie:
- 'Acc na szumie': Im wyższa wartość, tym bardziej odporny i zregularyzowany model.
- 'Test Loss': Im MNIEJSZA wartość, tym model rzadziej popełnia 'pewne siebie' błędy.
+----------------------+-----------------+-------------+--------------------------+
| Model Typ            | Acc na szumie   |   Test Loss |   Acc Train vs Test diff |
+======================+=================+=============+==========================+
| Float32 (Baza)       | 97.16%          |      0.0709 |                   1.13   |
+----------------------+-----------------+-------------+--------------------------+
| PTQ (Post-Training)  | 97.32%          |      0.0721 |                   1.1117 |
+----------------------+-----------------+-------------+--------------------------+
| QAT (Aware Training) | 97.60%          |      0.0619 |                   1.5667 |
+----------------------+-----------------+--------